# Day 1a — Biology intro & data setup

## Coding vs. non-coding DNA

A bacterial genome is a single long DNA string. Only part of it actually **codes for
protein** (a *CDS*, coding sequence — read in triplets called codons, each specifying one
amino acid). The rest is **non-coding**: intergenic regions, regulatory sequences, etc.

Our task this week: **given a short window of DNA, predict whether it's inside a CDS or not.**

This is a real, useful problem — gene finding is a foundational bioinformatics task, and
it's binary, which keeps things approachable for a first project.

## Look at the raw files first

No preprocessing yet. Every genome in `data/supervised/raw/` is a pair of files:
- a **FASTA** file: the raw genome sequence
- a **GFF** file: annotations — where genes/CDS regions are, on which strand

Let's look at both directly.

In [ ]:
import sys
sys.path.append("../src")

RAW_DIR = "../../data/supervised/raw"

In [ ]:
# peek at a raw genome FASTA file
with open(f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.fasta") as f:
    for _ in range(6):
        print(f.readline().rstrip())

In [ ]:
# peek at its GFF annotation file — note the CDS rows (columns: seqid, source,
# type, start, end, score, strand, phase, attributes)
with open(f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.gff") as f:
    lines = [l for l in f if not l.startswith("#")]
for l in lines[:8]:
    print(l.rstrip())

## From raw files to a labeled dataset

`src/build_dataset.py` is the provided script that turns this into something we can train
on. What it does, per genome:

1. Read all CDS intervals from the GFF (`type == "CDS"`, columns `start`/`end`/`strand`).
2. Slide fixed-length windows (200bp) **inside** CDS regions → label `1` (coding). If the
   CDS is on the `-` strand, it reverse-complements the window first, so the model always
   sees sequence in coding orientation.
3. Compute the **gaps** between CDS regions (everything not covered by any CDS) and slide
   the same windows there → label `0` (non-coding).
4. Balance classes per genome, shuffle, write one CSV per split.

It's already been run once (`data/supervised/processed/{train,val,test}.csv` exist). If you
want to change the window size or resampling, re-run it:

```bash
python ../src/build_dataset.py --raw_dir ../../data/supervised/raw \
    --out_dir ../../data/supervised/processed --window 200 --stride 200
```

In [ ]:
from data import load_all

splits = load_all("../../data/supervised/processed")
for name, df in splits.items():
    print(name, df.shape, "class balance:", df["label"].value_counts().to_dict())

splits["train"].head()

### Checkpoint

You should have `train`, `val`, `test` DataFrames, each with a `sequence` column (the raw
200bp window) and a `label` column (1 = coding, 0 = non-coding), roughly class-balanced.

Next: `01_kmer_and_cnn_baselines.ipynb` — build the first models on top of this data.